# Dub a video with IndicF5 — Colab

This notebook is **step 2 of 3**. It does one job: take the bundle your laptop
produced, run the voice model on a GPU, and hand you a results zip.

```
  1. your laptop     python -m src.cli ...            ->  tts_bundle.zip
  2. THIS NOTEBOOK   upload the zip, synthesize        ->  <job>_result.zip
  3. your laptop     python -m src.cli --from-stage import  ->  dubbed.mp4
```

Everything expensive is already done before you get here. Translation, timing
and segmentation ran on your laptop; this side only speaks the text it is
given, at the durations it is told.

**You need, before starting:**

| | |
|---|---|
| `tts_bundle.zip` | produced by step 1 on your laptop |
| a Hugging Face token | free, from huggingface.co/settings/tokens |
| access to `ai4bharat/IndicF5` | it is a gated repo — open the model page and accept the terms once |

Runtime is about **3–6 minutes on a T4** for a 60-second video, most of it
model download on the first run.

> **Colab vs Kaggle.** The only real differences are how you get files in and
> how the token is read. Colab lets you upload straight into the session;
> Kaggle needs the file attached as a Dataset first. Both are covered in their
> own notebook, and the synthesis in the middle is identical.

## 0. Turn on the GPU

**Runtime → Change runtime type → T4 GPU → Save.**

Run the cell. **Expected:** a line naming a Tesla T4. If it prints nothing or
errors, the runtime is still on CPU.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Your Hugging Face token

IndicF5 is a **gated** repo: you must be signed in, and you must have accepted
its terms at least once at <https://huggingface.co/ai4bharat/IndicF5>.

**On Colab, use the key icon (🔑) in the left sidebar** — this is where Colab
differs from Kaggle, which puts tokens under Add-ons → Secrets:

1. Click the **🔑** in the left sidebar
2. **Add new secret**
3. Name: `HF_TOKEN` — exactly that, it is case-sensitive
4. Value: your token from huggingface.co/settings/tokens (a `read` token is enough)
5. **Toggle "Notebook access" on** — this is the step people miss

The cell falls back to a hidden prompt if the secret is not there, so you can
also just paste the token when asked. The secret is better: it survives every
restart, and a pasted token does not.

**Expected:** `logged in as <your-username>`.

In [ ]:
from huggingface_hub import login, whoami

token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    pass

if not token:
    from getpass import getpass
    print("No HF_TOKEN secret found — paste a token instead (input is hidden).")
    token = getpass("HF token: ")

login(token=token)
print("logged in as", whoami()["name"])

## 2. Clone and install

Two installs, in this order — IndicF5 first, then our pins, so ours are the
ones that survive.

> **You will see a red dependency-conflict line about `f5-tts` and `numpy`.
> That is expected and is not a failure.** f5-tts declares a numpy ceiling
> inherited from an older scientific stack; librosa's numba needs a newer one.
> We deliberately override it, and IndicF5 runs correctly on numpy 2.2.
> `requirements-gpu.txt` explains it in full.

Takes about 2 minutes. **Expected:** the two `tail -2` lines, then the restart
instruction.

In [ ]:
import os

os.chdir("/content")
!rm -rf indic-dub-pipeline
!git clone -q -b test https://github.com/ayushk1233/indic-dub-pipeline.git
os.chdir("/content/indic-dub-pipeline")
!git log --oneline -1

!pip install -q git+https://github.com/ai4bharat/IndicF5.git 2>&1 | tail -2
!pip install -q -r requirements-gpu.txt 2>&1 | tail -2

print("\n" + "=" * 60)
print("INSTALLED. Now restart: Runtime -> Restart session.")
print("Then continue at section 3. Do NOT re-run this cell.")
print("=" * 60)

---

# ⛔ RESTART THE RUNTIME NOW — not optional

**Runtime → Restart session**, then carry on at section 3 below.

The install changed `numpy` and `transformers`. Anything Python already
imported in this session is still holding the *old* module objects, and the
failure does not show up here — it surfaces much later as a confusing
`ImportError` from inside the model loader, or as a model class that has
silently vanished.

**Do not re-run section 2 after restarting.** The clone and the packages are
already on disk. Everything from section 3 onward re-establishes what it needs.

Colab may offer **"Restart and run all"** — do not use it here. It would
re-run section 2 and put you back where you started.

---

## 3. Verify the environment (after the restart)

Run the cell. **Expected:** every line reports a version and the last line says
`environment OK`. If `numpy` is below 2.1 or `transformers` is 5.x, the restart
did not happen — restart and re-run this cell only.

In [ ]:
import os, sys

os.chdir("/content/indic-dub-pipeline")
sys.path.insert(0, "/content/indic-dub-pipeline")

import numpy, torch, transformers

print(f"numpy         {numpy.__version__:<12} (needs >=2.1,<2.3)")
print(f"torch         {torch.__version__:<12} cuda={torch.cuda.is_available()}")
print(f"transformers  {transformers.__version__:<12} (needs >=4.57,<5)")
print(f"gpu           {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")

assert torch.cuda.is_available(), "no GPU — Runtime > Change runtime type > T4, then restart"
assert numpy.__version__ >= "2.1", "stale numpy — you did not restart the runtime"

# librosa imports numba at module load, and numba hard-checks numpy with a
# ceiling that moves per numba version (0.60 caps it at <2.1, 0.62 at <2.4).
# Hosts preinstall old ones. Trigger that import here, where it costs a
# second, instead of inside the worker after the model has been fetched.
import numba, librosa  # noqa: F401
print(f"numba         {numba.__version__:<12} (needs >=0.62 for numpy 2.2)")
print(f"librosa       {librosa.__version__:<12} imported OK")

from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    from getpass import getpass
    login(token=getpass("HF token: "))

print("\nenvironment OK")

## 4. Upload your bundle

**This is the Colab-specific step.** Run the cell and a file picker appears —
choose the `tts_bundle.zip` your laptop produced. You can select more than one.

**Expected:** an upload progress bar, then one line per bundle naming its
segment count, language and job id.

> The upload lives only for this session. If the runtime disconnects you will
> need to upload again — the clone and packages survive a *restart*, but
> nothing survives a *disconnect*.

In [ ]:
import json, shutil, zipfile
from pathlib import Path

from google.colab import files

WORK = Path("/content/work")
WORK.mkdir(exist_ok=True)
BUNDLES = {}

uploaded = files.upload()

for name in uploaded:
    tmp = WORK / "_unzipped" / Path(name).stem
    shutil.rmtree(tmp, ignore_errors=True)
    with zipfile.ZipFile(name) as z:
        z.extractall(tmp)

    for manifest in tmp.rglob("manifest.json"):
        src = manifest.parent
        request = src / "request" / "synthesis_request.json"
        if not request.exists():
            continue
        job = json.loads(request.read_text(encoding="utf-8"))["job_id"]
        dest = WORK / f"{job}_bundle"
        shutil.rmtree(dest, ignore_errors=True)
        shutil.copytree(src, dest)
        BUNDLES[job] = dest

if not BUNDLES:
    raise SystemExit("No bundle in what you uploaded — expected a zip containing manifest.json")

for job, path in sorted(BUNDLES.items()):
    data = json.loads((path / "request" / "synthesis_request.json").read_text(encoding="utf-8"))
    print(f"{job:<12} {len(data['segments']):>3} segments   language={data['language']}   -> {path}")

## 5. Synthesize

This is the part that needs the GPU. The worker loads IndicF5 once, then walks
the bundle segment by segment.

**Expected output**, per segment: the text it was asked for, the duration it was
told to hit, and `done`. It writes its result file after *every* segment, so if
the session dies halfway the finished clips survive and re-running picks up a
complete file.

First run downloads roughly 2 GB of weights. Later runs in the same session
reuse the cache.

**If a segment says `failed`**, the run continues — one bad segment does not
lose the job. The error is recorded in `logs/errors.log` inside the bundle.

In [ ]:
for job, path in sorted(BUNDLES.items()):
    print(f"\n{'=' * 24} {job} {'=' * 24}")
    !python -m colab.indicf5_worker --bundle {path}

## 6. Download the results

Run the cell. **Expected:** one browser download per bundle, a few MB each.

If your browser blocks the automatic download, allow pop-ups for
`colab.research.google.com` and re-run — or open the file browser (📁 in the
left sidebar) and download from `/content/work/` by hand.

Save them somewhere you can find — step 3 on your laptop needs them.

In [ ]:
import shutil

from google.colab import files

for job, path in sorted(BUNDLES.items()):
    archive = shutil.make_archive(str(WORK / f"{job}_result"), "zip", path / "output")
    print(f"{job:<12} {Path(archive).stat().st_size / 1e6:>6.1f} MB")
    files.download(archive)

## 7. Listen before you leave (optional, recommended)

Worth 30 seconds. Every timing metric can read perfectly green while the audio
is wrong, so the only check that catches a bad run early is your ear.

Play a few clips. They should say the text printed above them, in the
speaker's voice, with nothing invented at the start.

In [ ]:
import json
from IPython.display import Audio, display

job = sorted(BUNDLES)[0]        # change this to hear another bundle
path = BUNDLES[job]

asked = {s["segment_id"]: s["text"] for s in json.loads(
    (path / "request" / "synthesis_request.json").read_text(encoding="utf-8"))["segments"]}
result = json.loads((path / "output" / "synthesis_result.json").read_text(encoding="utf-8"))

for segment in result["segments"][:5]:
    index = segment["segment_id"]
    print(f"[{index:>2}] {segment['status']}  {segment.get('duration', 0):.2f}s")
    print(f"     {asked[index]}")
    if segment["status"] == "done":
        display(Audio(str(path / segment["audio_path"])))

---

## Back to your laptop

Unzip each result into its job's bundle directory, then run the third command:

```bash
unzip -o ~/Downloads/demo_result.zip -d artifacts/demo/tts_bundle/output/

python -m src.cli \
  --input myvideo.mp4 \
  --job-id demo \
  --target-lang hi \
  --from-stage import
```

That assembles the clips onto the original timeline and writes
`artifacts/demo/dubbed.mp4`.

---

## Troubleshooting

**`OSError: You are trying to access a gated repo`**
You have a token but have not accepted the model terms. Open
<https://huggingface.co/ai4bharat/IndicF5> while signed in, accept, re-run
section 3.

**`SecretNotFoundError` or the token prompt keeps appearing**
The secret exists but **Notebook access** is toggled off. Open the 🔑 panel and
turn it on for this notebook. Creating the secret and granting access are two
separate switches.

**`ImportError` mentioning `transformers` or a missing model class**
You did not restart after section 2, or you used "Restart and run all", which
re-ran the install. Runtime → Restart session, then start again at section 3.

**A red `pip` line about `f5-tts` requiring `numpy<=1.26.4`**
Expected. Not a failure. See the note in section 2.

**Everything disappeared and `BUNDLES` is undefined**
The runtime disconnected, which wipes `/content`. Re-run from section 2,
including the upload. Colab disconnects free sessions after roughly 90 minutes
idle.

**`CUDA out of memory`**
Runtime → Restart session and re-run from section 3. If it persists, Colab has
given you a busier GPU than usual — the Kaggle notebook in this same folder
does the identical job.

**The download never starts**
Allow pop-ups for `colab.research.google.com`, or open 📁 in the left sidebar
and download from `/content/work/` manually.